In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

# Qiskit & Optimization imports
from qiskit_aer import AerSimulator
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.converters import QuadraticProgramToQubo

In [3]:
tickers = ['BBCA.JK', 'TLKM.JK', 'ASII.JK', 'BMRI.JK']
data = yf.download(tickers, start='2024-01-01', end='2025-01-01')['Close']

# Hitung expected return tahunan dan matriks kovariansi
daily_returns = data.pct_change().dropna()
mu = daily_returns.mean().values * 252       # Expected return
sigma = daily_returns.cov().values * 252      # Covariance matrix

print("Expected Returns (μ):", np.round(mu, 4))
print("\nCovariance Matrix (Σ):\n", np.round(sigma, 4))

[*********************100%***********************]  4 of 4 completed

Expected Returns (μ): [-0.0146  0.0822  0.0229 -0.3059]

Covariance Matrix (Σ):
 [[0.0676 0.018  0.0249 0.0252]
 [0.018  0.0496 0.0361 0.0159]
 [0.0249 0.0361 0.0928 0.0212]
 [0.0252 0.0159 0.0212 0.0876]]


In [4]:
num_assets = len(tickers)
q = 0.5       # Faktor risiko: 0 = agresif, 1 = defensif (risk-averse)
budget = 2    # Jumlah saham yang ingin dipilih

# Inisialisasi program kuadratik
qp = QuadraticProgram(name="Portfolio_Optimization")

# 1. Buat variabel keputusan biner untuk masing-masing saham (0: tidak beli, 1: beli)
for ticker in tickers:
    qp.binary_var(name=ticker)

# 2. Rumuskan Objective Function: Minimalkan Risiko - Maksimalkan Return
# Min: q * (x^T * Sigma * x) - (mu^T * x)
linear_terms = {tickers[i]: -float(mu[i]) for i in range(num_assets)}

quadratic_terms = {}
for i in range(num_assets):
    for j in range(num_assets):
        val = q * float(sigma[i, j])
        if val != 0:
            quadratic_terms[(tickers[i], tickers[j])] = val

qp.minimize(linear=linear_terms, quadratic=quadratic_terms)

# 3. Batasan Budget (Constraint): Jumlah saham yang dipilih harus sama dengan budget
qp.linear_constraint(
    linear={ticker: 1 for ticker in tickers},
    sense="==",
    rhs=budget,
    name="budget_constraint"
)

print(qp.prettyprint())

Problem name: Portfolio_Optimization

Minimize
  0.04639473068198212*ASII.JK^2 + 0.021239584611248235*ASII.JK*BMRI.JK
  + 0.024935513205662233*BBCA.JK*ASII.JK + 0.03380252618324989*BBCA.JK^2
  + 0.025154558478986955*BBCA.JK*BMRI.JK + 0.01803800489177795*BBCA.JK*TLKM.JK
  + 0.04378470126811595*BMRI.JK^2 + 0.03612611068723488*TLKM.JK*ASII.JK
  + 0.015868110610224355*TLKM.JK*BMRI.JK + 0.024811995209398318*TLKM.JK^2
  - 0.022904082158727225*ASII.JK + 0.014637975364398781*BBCA.JK
  + 0.30586975406221606*BMRI.JK - 0.08224700378737212*TLKM.JK

Subject to
  Linear constraints (1)
    ASII.JK + BBCA.JK + BMRI.JK + TLKM.JK == 2  'budget_constraint'

  Binary variables (4)
    BBCA.JK TLKM.JK ASII.JK BMRI.JK

